# Klasyfikacja obrazów za pomocą CNN


* rozszerzanie danych (*data augmentation*)
* korzytsanie z wytrenowanych modeli (*model zoo*)
* ekstrakcja cech z wytrenowanych modeli (*transfer learning*)


In [ ]:
import matplotlib.pyplot as plt 
import numpy as np
import os
import tensorflow as tf
print('Wersja TensorFlow:', tf.__version__)

## Dane: psy i koty

Dane: zdjęcia psów i kotów  
Problem: rozpoznawanie zwięrząt (klasyfikacja)  
URL: https://www.fizyka.umk.pl/~grochu/nn/data/cats_and_dogs_filtered.zip

Pobierzmy dane i rozpakujmy w katalogu `./dane/`

In [ ]:
import requests
import zipfile
import io

URL = "https://www.fizyka.umk.pl/~grochu/nn/data/cats_and_dogs_filtered.zip"

print("Pobieram dane...")
response = requests.get(URL)

print("Rozpakowuję plik ZIP...")
with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
    zip_ref.extractall(path='./dane/')

## Dane uczące

Dane zawierają 3000 zdjęć podzielonych na zbiór uczący (`train`) i walidayjcny (`validation`)  
Dwie klasy: `dogs`, `cats`   
Zbiór uczący: 1000 zdjęć psów i 1000 kotów  
Zbiór walidacyjny: 500 psów i 500 kotów  

Struktura plików:
```
cats_and_dogs_filtered/
    train/
        dogs/
            dog001.jpg
            dog002.jpg
            ...
        cats/
            cat001.jpg
            cat002.jpg
            ...
    validation/
        dogs/
            dog001.jpg
            dog002.jpg
            ...
        cats/
            cat001.jpg
            cat002.jpg
            ...
```

## Import danych

In [ ]:
PATH = 'dane/cats_and_dogs_filtered'
train_dir = os.path.join(PATH, 'train')
validation_dir = os.path.join(PATH, 'validation')

print('Liczba obrazów treningowych kotów:',  len(os.listdir(os.path.join(train_dir, 'cats'))))
print('Liczba obrazów treningowych psów:',   len(os.listdir(os.path.join(train_dir, 'dogs'))))
print('Liczba obrazów walidacyjnych kotów:', len(os.listdir(os.path.join(validation_dir, 'cats'))))
print('Liczba obrazów walidacyjnych psów:',  len(os.listdir(os.path.join(validation_dir, 'dogs'))))

Wyświetlmy kilka przykładowych obrazów z obu klas

In [ ]:
n_img = 4  # Liczba obrazów do wyświetlenia

fig, axs = plt.subplots(2, 4, figsize=(16, 8))
for i, label in enumerate(['cats', 'dogs']):
    files = os.listdir(train_dir + '/' + label)
    for j, fname in enumerate(np.random.permutation(files)[:n_img]):
        img = plt.imread(os.path.join(train_dir, label, fname))
        axs[i, j].imshow(img)
        axs[i, j].set_title(fname)
plt.show()

## Wczytywanie danych

* [ft.data.Dataset](https://www.tensorflow.org/api_docs/python/tf/data/Dataset) <br> klasa do tworzenia sekwencji danych, która może być użyta do trenowania modelu

* [tf.keras.preprocessing.image_dataset_from_directory](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/image_dataset_from_directory) <br> funkcja do tworzenia sekwencji danych (`tf.data.Dataset`) z katalogu zawierającego obrazy


In [ ]:
img_size = (150, 150)   # docelowy rozmiar obrazów
batch_size  = 32        # rozmiar paczki danych uczących

train_dataset = tf.keras.utils.image_dataset_from_directory(train_dir,
                                                            shuffle=True,
                                                            batch_size=batch_size,
                                                            image_size=img_size)

Analogoczny generator danych dla zbioru walidacyjnego

In [ ]:
validation_dataset = tf.keras.utils.image_dataset_from_directory(validation_dir,
                                                                 shuffle=True,
                                                                 batch_size=batch_size,
                                                                 image_size=img_size)

## Przykładowa próbka z generatora danych

* `iter` - tworzy iterator dla generatora danych
* `next` - pobiera pierwszą paczkę danych z iteratora
* `X_batch` - paczka obrazów, `y_batch` - odpowiadające etykiety
* `X_batch.shape` - rozmiar paczki danych (liczba obrazów, wysokość, szerokość, liczba kanałów)
* `y_batch` - tablica etykiet (0 dla kotów, 1 dla psów)

In [ ]:
X_batch, y_batch = next(iter(train_dataset))

print(f'Kształt X_batch {X_batch.shape},  y_batch {y_batch.shape}')

image = X_batch.numpy().astype("uint8")[0, :, :, :]
label = train_dataset.class_names[y_batch[0]]   
plt.imshow(image)
plt.title(f'Klasa {label}');

## Kolejkowanie danych

* [prefetch()](https://www.tensorflow.org/api_docs/python/tf/data/Dataset#prefetch) - metoda do kolejkowania danych, która pozwala na przygotowanie następnej paczki danych podczas trenowania modelu na bieżącej paczce
* `AUTOTUNE` - automatyczne dostosowanie rozmiaru bufora do dostępnych zasobów systemowych

In [ ]:
train_dataset = train_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

## Model CNN klasyfikacji

Zbudujmy model CNN składający się z typowych elementów 

* warstwa wejściowa ``[150, 150, 3]``
* [Rescaling](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Rescaling) - warstwa normalizująca wartości pikseli do zakresu [0, 1]  (zob. inne warstwy [preprocessing layers](https://keras.io/api/layers/preprocessing_layers/image_preprocessing/))
* warstwy Conv2D, MaxPool2D, Dense 
* warstwa wyjsciowa 1 neuron ``sigmoid``, klasyfikacja binarna etykiety 0 lub 1

In [ ]:
from tensorflow.keras import layers, Model

tf.keras.backend.clear_session()

inputs = layers.Input(shape=(150, 150, 3))
x = layers.Rescaling(1./255, offset=-0.5)(inputs)
x = layers.Conv2D(16, (3, 3), activation='relu')(x)
x = layers.MaxPool2D(2)(x)
x = layers.Conv2D(32, (3, 3), activation='relu')(x)
x = layers.MaxPool2D(2)(x)
x = layers.Dense(100, activation='relu')(x)
x = layers.Flatten()(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

## Uczenie sieci 

Konfiguracja treningu:
* funkcja kosztu ``binary_crossentropy`` do klasyfikacji binarnej
* metryka ``binary_accuracy`` - miara poprawności klasyfikacji binarnej


In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(loss='binary_crossentropy', optimizer=Adam(), metrics=['binary_accuracy'])

model.fit(train_dataset, epochs=10, validation_data=validation_dataset)

In [ ]:
history = model.history.history

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(history['binary_accuracy'], label='Training Accuracy')
plt.plot(history['val_binary_accuracy'], label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Aaccuracy')

plt.subplot(2, 1, 2)
plt.plot(history['loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.xlabel('Epochs');

## Wizualizacja aktywacji map cech

In [ ]:
from tensorflow.keras.preprocessing.image import img_to_array, load_img

layers = [ model.layers[2].output, model.layers[4].output ] # wybrane warstwy splotowe 
vis_model = Model(model.inputs, layers)                     # model z dwoma wyjściami

img = load_img('dane/cats_and_dogs_filtered/train/cats/cat.1.jpg', target_size=(150, 150))  
plt.imshow(img)

x = np.array([ img_to_array(img) ])   # normalizacja i utworzenie paczki
feature_maps = vis_model.predict(x)          # predykcja

print(f'Input shape: {x.shape}')
print(f'Output 1 shape: {feature_maps[0].shape}')
print(f'Output 2 shape: {feature_maps[1].shape}')

In [ ]:
for layer, feature_map in zip(layers, feature_maps):
    
    # mapa cech (feature_map) ma kształt (1, size, size, n_features)
    size, n_features = feature_map.shape[1], feature_map.shape[-1]
    display_grid = np.zeros((size, size * n_features))
    for i in range(n_features):
        x = feature_map[0, :, :, i] 
        display_grid[:, i * size : (i + 1) * size] = x 
    
    plt.figure(figsize=(20, 20 / n_features))
    plt.title(layer.name)
    plt.imshow(display_grid, aspect='auto', cmap='Reds')

### Ćwiczenie: warstwa Dropout

Dodaj warstwę dropout na wyjsciu warstw splotowych i gęstych aby ograniczyć przeuczenie.   
Wytrenuj model i porównaj uzyskany wynik z poprzednią architekturą bez warstwy dropout.  
Zobacz jaki wpływ na wynik ma wartość argumentu ``rate``.

Dokumentacja: [tf.keras.layers.Dropout(rate)](https://keras.io/api/layers/regularization_layers/dropout/)





## Data augmentation

Dodajmy losowe transformacje obrazów, które spowodują sztuczne zwielokrotnienie zbioru uczacego - w praktyce żaden obraz wejściowy nie będzie się powtarzał.  

* [Image Augmentation Layers](https://keras.io/api/layers/preprocessing_layers/image_augmentation/)


Warto zaznaczyć: te wartswy będą aktywyłącznie w czasie treningu (`Model.fit`). Nie będą aktywne podczas predykcji (`Model.evaluate`, `Model.predict`). W tym podczas wyznaczania błedu walidacyjnego.

In [ ]:
from tensorflow.keras import layers, Sequential

data_augmentation = Sequential([
  layers.RandomFlip('horizontal'),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.2),
  layers.RandomTranslation(0.2, 0.2)
])


## Przykład transfromacji

In [ ]:
img = plt.imread('dane/cats_and_dogs_filtered/train/cats/cat.1.jpg')
img = img.reshape((1,) + img.shape)
augmented_image = data_augmentation(img)

fig, ax = plt.subplots(1, 2, figsize=(8, 8))
ax[0].imshow(img[0, :, :, :]/255.)
ax[0].set_title("Original")
ax[1].imshow(augmented_image[0, :, :, :]/255.)
ax[1].set_title("Augmented")
plt.show()

### Ćwiczenie

Zbuduj ponowanie model z warstwami dropout używając rozszerzonego zbioru treningowego i porównaj wyniki

## Zapis modelu 

Zapis modelu do formatu `keras`, `h5` lub `tf` 
```python
Model.save(filepath, overwrite=True, save_format=None)
```

Odczyt modelu
```python
tf.keras.models.load_model(filepath)

```


Dokumentacja: [Save, serialize, and export models](https://keras.io/2/api/models/model_saving_apis/model_saving_and_loading/)



In [ ]:
# zapis do formatu pliku HDF5 
model.save('model.h5')

In [ ]:
model2 =  tf.keras.models.load_model("model.h5")
model2.summary()

## Wytrenowane modele Keras 

Modele dostepne w Keras https://keras.io/api/applications/

* VGG16, VGG19
* ResNet50, ResNet101
* InceptionV3, Xception


## InceptionV3

Model [InceptionV3](https://keras.io/api/applications/inceptionv3/) wytrenowany na zbiorze ImageNet 1.4M obrazów podzielonych na 1000 klas.

Uwaga: przy pierwszym uruchomieniu model jest pobierany do kotalogu z modelami (w linuksie domyślny katalog modeli to ``~/.keras/models/``)

Więcej o architekturze Inception V3: [Inception V3 CNN Architecture Explained ](https://medium.com/@AnasBrital98/inception-v3-cnn-architecture-explained-691cfb7bba08)

In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3

model = InceptionV3(weights='imagenet')

model.summary()

## Klasyfikacja obrazu za pomocą InceptionV3

Użycie konkretnego modelu może wymagać specyficznych transformacji danych wejściowych oraz odkodowania wyjścia  
* funkcja  [tf.keras.applications.inception_v3.preprocess_input()](https://www.tensorflow.org/api_docs/python/tf/keras/applications/inception_v3/preprocess_input) - przetwarza obrazy aby pasowały do wejścia sieci `inception_v3` 
* funkcja [tf.keras.applications.inception_v3.decode_predictions()](https://www.tensorflow.org/api_docs/python/tf/keras/applications/inception_v3/decode_predictions) - odkodowuje etykiety klas ImageNet z sygnału wyjściowego sieci `imagenet_v3` (domyślnie zwraca `top=5` predykcji)

In [ ]:
from tensorflow.keras.preprocessing import image
import pandas as pd

model = InceptionV3(weights='imagenet')

img_path = 'dane/cats_and_dogs_filtered/train/cats/cat.1.jpg'
img = image.load_img(img_path, target_size=(299, 299))
# plt.imshow(img)

x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = tf.keras.applications.inception_v3.preprocess_input(x)

preds = model.predict(x)
print(f'Preds shape: {preds.shape}')

decoded = tf.keras.applications.inception_v3.decode_predictions(preds, top=5)[0]
print(f'Decoded: {decoded}')

dt = pd.DataFrame(decoded, columns=['Name', 'Label', 'Prob'])
dt.plot.barh(x='Label', y='Prob');

### Ćwiczenie

Użyj sieci InceptionV3 do predykcji obiektu na dowolnym zdjęciu np. pobranym z internetu. 

## Transfer learning

Ekstrakcja cech z użyciem początkowych warstw wytrenowanych modeli i douczanie na nowym problemie
* usuwamy warstwę klasyfikujacą modelu wytrenowanego na duzym zbiorze danych (`include_top=False`)
* zamrażamy wagi (ustawiamy ``trainable=False``) aby nie były optylizowane w czasie douczania
* dodajemy własne warstwy oraz warstwę klasyfikującą
* uczymy na nowych danych 






In [ ]:
tf.keras.backend.clear_session()

pre_trained_model = InceptionV3(
        weights="imagenet",  
        include_top=False,           # pobiera model z usuniętą ostatnią warstwą
        input_shape=(150, 150, 3),   # dla sieci w roli ekstraktora cech ustalmy inny rozmiar wejscia
    )

pre_trained_model.summary()

## Zamrożenie wag

Dowolne warstwy mogą być wyłączone z procesu uczenia (sygnał będzie nadal przez nie przetwarzany)

In [ ]:
for layer in pre_trained_model.layers:
      layer.trainable = True

To samo można wykonac dla całej sieci

In [ ]:
pre_trained_model.trainable = False
pre_trained_model.summary()

## Wybór warstwy do ekstrakcji cech

* najczęściej wybiera się ostatnią warstwę ale możemy wybrać inną, wcześniejszą

In [ ]:
last_layer = pre_trained_model.get_layer('mixed10')

print(f'last layer output shape: {last_layer.output_shape}')
last_output = last_layer.output


base_model = Model(inputs=pre_trained_model.input, outputs=last_output)
base_model.summary()

## Nowa warstwa ucząca i klasyfikująca

Głowa klasyfikująca:
* wypłaszczenie mapy cech ``Flatten()`` do postaci wektora lub ``GlobalAveragePooling2D()``, który wyznaczy średnią z każdej 3x3 ostatniej warstwy  
* dodanie warstw gęstych oraz warstwy klasyfikującej 

In [ ]:
from tensorflow.keras.layers import Flatten, Dense

inputs = tf.keras.Input(shape=(150, 150, 3))

# rozszerzanie danych
x = data_augmentation(inputs)

# preprocessing
x = layers.Rescaling(1./255., offset=-0.5)(x)

# ekstrakcja cech
x = pre_trained_model(x, training=False)

# głowa klasyfikacyjna
x = layers.GlobalAveragePooling2D()(x)
x = Dense(1, activation='sigmoid')(x)

model = Model(inputs, x)
model.summary()

## Douczanie (fine tuning)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)

loss0, accuracy0 = model.evaluate(validation_dataset)
print(f'initial loss: {loss0:.2f}, initial accuracy: {accuracy0:.2f}')

In [ ]:
model.fit(train_dataset, epochs=5, validation_data=train_dataset)

loss1, accuracy1 = model.evaluate(validation_dataset)
print(f'final loss: {loss1:.2f}, final accuracy: {accuracy1:.2f}')